Implement First Visit Monte Carlo prediction algorithm using Python and apply it to a simple environment. Display the learned policy and final Q-values.

In [1]:
import numpy as np
import random

n_states = 16
n_actions = 4
gamma = 0.9
episodes = 5000

action_symbols = ['U', 'D', 'L', 'R']
Q = np.zeros((n_states, n_actions))
counts = np.zeros((n_states, n_actions))

In [2]:
def choose_action(state, epsilon=0.1):
    if random.random() < epsilon:
        return random.randint(0, 3)
    return np.argmax(Q[state])

def step(state, action):
    row, col = divmod(state, 4)

    if action == 0:
        row = max(0, row - 1)
    elif action == 1:
        row = min(3, row + 1)
    elif action == 2:
        col = max(0, col - 1)
    elif action == 3:
        col = min(3, col + 1)

    next_state = row * 4 + col

    if next_state == 15:
        reward = 10
        done = True
    else:
        reward = -1
        done = False

    return next_state, reward, done

In [3]:
for episode in range(episodes):
    state = 0
    trajectory = []
    done = False
    steps = 0

    while not done and steps < 100:
        action = choose_action(state)
        next_state, reward, done = step(state, action)

        trajectory.append((state, action, reward))
        state = next_state
        steps += 1

    G = 0
    visited = set()

    for t in reversed(range(len(trajectory))):
        state, action, reward = trajectory[t]
        G = gamma * G + reward

        if (state, action) not in visited:
            visited.add((state, action))
            counts[state, action] += 1
            Q[state, action] += (G - Q[state, action]) / counts[state, action]

In [4]:
print("Learned Policy (First-Visit MC):")
for state in range(n_states):
    if state == 15:
        print(" G ", end=" ")
    else:
        best_action = np.argmax(Q[state])
        print(f" {action_symbols[best_action]} ", end=" ")

    if state % 4 == 3:
        print()

print("\nFinal Q-values:")
print(np.round(Q, 2))

Learned Policy (First-Visit MC):
 D   L   D   D  
 R   R   D   D  
 U   R   R   D  
 U   R   R   G  

Final Q-values:
[[-0.55  1.13 -0.35 -1.46]
 [-1.93 -1.51 -0.3  -1.21]
 [ 1.02  3.78 -3.76 -5.66]
 [-1.32  1.62 -3.09 -7.2 ]
 [-0.36 -0.84  0.8   2.57]
 [-1.26  2.77  0.82  4.09]
 [ 1.74  5.93  2.13  4.42]
 [-4.31  7.67  4.01  1.83]
 [ 0.83 -2.29 -6.29 -2.5 ]
 [ 2.46 -4.06 -3.65  5.83]
 [ 3.99  7.38  2.97  7.87]
 [ 4.05 10.    5.88  7.76]
 [-1.02 -6.93 -9.74 -7.49]
 [-5.44 -0.93 -6.59  5.52]
 [ 4.6   3.5   1.83 10.  ]
 [ 0.    0.    0.    0.  ]]
